# Day 12 · Exercise 1: benchmark_search

**What you'll build:** `brute_force_search(query_embedding: list[float], index: list[dict], top_k: int) -> list[dict]` — a function that scores every entry in an in-memory index by cosine similarity to a query vector and returns the top-k results sorted by descending score.

**Why it matters:** Implementing brute-force search yourself makes O(n) scaling tangible — you can then measure it directly and understand exactly why dedicated vector databases with ANN indexing exist.

> **No Ollama needed** for this exercise — it uses synthetic random vectors.

## Your Implementation

In [ ]:
import math


def brute_force_search(
    query_embedding: list[float],
    index: list[dict],
    top_k: int,
) -> list[dict]:
    """Search an in-memory index by cosine similarity and return the top-k results.

    Each entry in `index` is a dict with at least two keys:
      - "text"      (str)        — the original document text
      - "embedding" (list[float]) — the document's vector representation

    The function computes cosine similarity between `query_embedding` and every
    stored embedding, then returns the `top_k` entries with the highest scores,
    sorted in descending order.  This is O(n) in the number of index entries.

    Args:
        query_embedding: A list of floats representing the query vector.
            Must have the same dimensionality as the stored embeddings.
        index: A list of dicts, each containing "text" and "embedding" keys.
        top_k: How many results to return.

    Returns:
        A list of dicts (length <= top_k), each containing:
          - "text"  (str)   — the document text from the index entry
          - "score" (float) — cosine similarity in [-1.0, 1.0], higher is more similar
        Sorted by score descending (best match first).

    Example:
        >>> index = [
        ...     {"text": "cats", "embedding": [1.0, 0.0]},
        ...     {"text": "dogs", "embedding": [0.0, 1.0]},
        ... ]
        >>> results = brute_force_search([1.0, 0.0], index, top_k=1)
        >>> results[0]["text"]
        'cats'
        >>> round(results[0]["score"], 4)
        1.0
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 5 automated checks and shows ✅ / ❌ for each.

In [ ]:
import math
import random
import time

_PASS, _FAIL = '✅', '❌'


def _run_checks():
    score, total = 0, 5

    # ── Check 1: function exists and is callable ────────────────
    try:
        assert callable(brute_force_search), 'brute_force_search is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return  # later checks would crash without a working function

    # ── Check 2: returns a list of dicts with correct keys ──────
    try:
        _index = [
            {"text": "cats", "embedding": [1.0, 0.0]},
            {"text": "dogs", "embedding": [0.0, 1.0]},
            {"text": "fish", "embedding": [0.7, 0.7]},
        ]
        _result = brute_force_search([1.0, 0.0], _index, top_k=2)
        assert isinstance(_result, list), f'expected list, got {type(_result).__name__}'
        assert len(_result) == 2, f'expected 2 results, got {len(_result)}'
        assert all(isinstance(r, dict) for r in _result), 'every result must be a dict'
        assert all('text' in r and 'score' in r for r in _result), \
            'each result dict must have "text" and "score" keys'
        print(f'{_PASS} Check 2/{total}: returns a list of {len(_result)} dicts with "text" and "score"')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
        return  # structure is broken; remaining checks would crash

    # ── Check 3: top result is correct (exact match) ────────────
    try:
        _index2 = [
            {"text": "cats", "embedding": [1.0, 0.0]},
            {"text": "dogs", "embedding": [0.0, 1.0]},
        ]
        _r = brute_force_search([1.0, 0.0], _index2, top_k=1)
        assert _r[0]['text'] == 'cats', f'expected "cats" as top result, got "{_r[0]["text"]}"'
        assert abs(_r[0]['score'] - 1.0) < 1e-6, \
            f'expected score ≈ 1.0 for identical vectors, got {_r[0]["score"]}'
        print(f'{_PASS} Check 3/{total}: top result is correct ("cats", score ≈ 1.0)')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # ── Check 4: results are sorted descending by score ─────────
    try:
        _index3 = [
            {"text": "a", "embedding": [1.0, 0.0, 0.0]},
            {"text": "b", "embedding": [0.6, 0.8, 0.0]},
            {"text": "c", "embedding": [0.0, 0.0, 1.0]},
            {"text": "d", "embedding": [0.5, 0.5, 0.7]},
        ]
        _query = [1.0, 0.0, 0.0]
        _res = brute_force_search(_query, _index3, top_k=4)
        _scores = [r['score'] for r in _res]
        assert _scores == sorted(_scores, reverse=True), \
            f'results not sorted descending: scores were {_scores}'
        print(f'{_PASS} Check 4/{total}: results are sorted descending by score')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    # ── Check 5: O(n) scaling — larger index takes more time ────
    try:
        random.seed(42)
        DIM = 128

        def _rand_vec(d):
            return [random.gauss(0, 1) for _ in range(d)]

        def _make_index(n, d):
            return [{"text": f"doc_{i}", "embedding": _rand_vec(d)} for i in range(n)]

        _q = _rand_vec(DIM)
        _small = _make_index(500, DIM)
        _large = _make_index(5_000, DIM)

        _t0 = time.perf_counter()
        brute_force_search(_q, _small, top_k=5)
        _t_small = time.perf_counter() - _t0

        _t1 = time.perf_counter()
        brute_force_search(_q, _large, top_k=5)
        _t_large = time.perf_counter() - _t1

        assert _t_large > _t_small, \
            f'larger index ({len(_large)}) should be slower than smaller ({len(_small)})'
        _ratio = _t_large / max(_t_small, 1e-9)
        print(
            f'{_PASS} Check 5/{total}: O(n) scaling confirmed '
            f'(10× corpus → {_ratio:.1f}× slower, small={_t_small*1000:.1f} ms, '
            f'large={_t_large*1000:.1f} ms)'
        )
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 5/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {score}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')


_run_checks()

## Bonus Challenge

Extend `brute_force_search` to accept an optional `threshold: float = -1.0` parameter and filter out any results whose score falls below it. Then time it on a 10,000-entry index with `threshold=0.5` — does filtering improve query time? (Spoiler: it doesn't. Think about why, and what that tells you about where ANN indexing's speed-up actually comes from.)

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import math


def brute_force_search(
    query_embedding: list[float],
    index: list[dict],
    top_k: int,
) -> list[dict]:
    def _cosine(a: list[float], b: list[float]) -> float:
        dot = sum(x * y for x, y in zip(a, b))
        mag_a = math.sqrt(sum(x * x for x in a))
        mag_b = math.sqrt(sum(x * x for x in b))
        if mag_a == 0.0 or mag_b == 0.0:
            return 0.0
        return dot / (mag_a * mag_b)

    scored = [
        {"text": entry["text"], "score": _cosine(query_embedding, entry["embedding"])}
        for entry in index
    ]
    scored.sort(key=lambda e: e["score"], reverse=True)
    return scored[:top_k]
```

**Why this works:** The inner `_cosine` helper computes the standard cosine similarity — dot product divided by the product of magnitudes — returning a value in [-1, 1] where 1.0 means identical direction. The list comprehension scores every entry in `index` without exception, which is exactly what makes this O(n): there is no early exit, no pruning, and no way to skip entries without inspecting them. Sorting by score descending and slicing to `top_k` gives the most similar results. The O(n) cost is unavoidable here, and that is the whole point — this implementation is the baseline that motivates ANN indexing in the rest of Day 12.
</details>